| Sector | Number of States | States |
|------|------|------|
Construction | 4 | Arizona, Louisiana, Texas, Utah
Energy | 14 | Alaska, Arkansas, Idaho, Mississippi, Montana, Nebraska, New Mexico, North Dakota, Oklahoma, Oregon, South Dakota, Vermont, West Virginia, Wyoming
Finance | 8 | Connecticut, Delaware, Minnesota, New Jersey, New York, North Carolina, Pennsylvania, Rhode Island
Manufacturing | 13 | Alabama, Illinois, Indiana, Iowa, Kansas, Kentucky, Michigan, Missouri, New Hampshire, Ohio, South Carolina, Tennessee, Wisconsin
Tech | 7 | California, Colorado, Georgia, Maryland, Massachusetts, Virginia, Washington
Tourism | 4 | Florida, Hawaii, Maine, Nevada



### Location Quotient (LQ)

The Location Quotient measures whether a sector is **more concentrated in a state than in the national economy**.

We define:

- $i$ = state index  
- $j$ = state index used when summing across all states  
- $s$ = sector (industry)  
- $k$ = sector index used when summing across all sectors  
- $C_{i,s}$ = compensation in sector $s$ in state $i$

---

### Step 1 — State sector share

First compute the share of sector $s$ within the economy of state $i$.

$$
\text{StateShare}_{i,s} =
\frac{C_{i,s}}{\sum_{k} C_{i,k}}
$$

Explanation:

- $C_{i,s}$ is compensation in sector $s$ in state $i$  
- $\sum_k C_{i,k}$ is the **sum of compensation across all sectors $k$ within state $i$**

Thus the denominator represents **total compensation across all industries in that state**.

This measures the **importance of sector $s$ within the state's economy**.

---

### Step 2 — National sector share

Next compute the share of sector $s$ in the national economy.

$$
\text{NationalShare}_{s} =
\frac{\sum_{j} C_{j,s}}
{\sum_{j}\sum_{k} C_{j,k}}
$$

Explanation:

- $\sum_j C_{j,s}$ is **total U.S. compensation in sector $s$ across all states**
- $\sum_j \sum_k C_{j,k}$ is **total compensation across all sectors and all states**

Thus the denominator represents **total national compensation across the entire economy**.

This measures the **importance of sector $s$ in the national economy**.

---

### Step 3 — Location Quotient

The Location Quotient compares the state's sector share to the national sector share.

$$
LQ_{i,s} =
\frac{\text{StateShare}_{i,s}}
{\text{NationalShare}_{s}}
$$

Substituting the expressions above:

$$
LQ_{i,s} =
\frac{\frac{C_{i,s}}{\sum_k C_{i,k}}}
{\frac{\sum_j C_{j,s}}{\sum_j \sum_k C_{j,k}}}
$$

---

### Interpretation

- $LQ_{i,s} > 1$ → sector $s$ is **more concentrated in state $i$ than nationally**  
- $LQ_{i,s} = 1$ → sector has the **same concentration as the national average**  
- $LQ_{i,s} < 1$ → sector is **less concentrated than average**

For each state $i$, the **dominant sector** is defined as the sector $s$ with the **largest $LQ_{i,s}$**.

### Data Source and Preparation

The industry data used in this analysis comes from the **Bureau of Economic Analysis (BEA) Regional Economic Accounts**, specifically the table:

**SAINC6N — Compensation of Employees by NAICS Industry**

The data was downloaded from the BEA iTables system with the following selections:

- Geography: **All 50 U.S. states**  
- Year: **2023**  
- Units: **Levels (thousands of dollars)**  
- Industry detail: **Major NAICS industry sectors**

The original dataset contained compensation of employees across **19 industry categories**. Two sectors were removed:

- Government and government enterprises  
- Real estate and rental and leasing  

These sectors were excluded because they tend to dominate compensation totals in many states and do not clearly represent the underlying structure of private industry. After removing them, the cleaned dataset contained **17 industries**.

For the specialization analysis, several additional industries were excluded because they represent **large local service sectors** that appear in nearly every state and primarily scale with population size rather than regional specialization. The industries removed at this stage were:

- Wholesale trade  
- Retail trade  
- Transportation and warehousing  
- Administrative and support and waste management and remediation services  
- Educational services  
- Health care and social assistance  

These sectors largely reflect **local demand (e.g., healthcare, retail, education)** and therefore do not meaningfully differentiate state economic structures. Removing them allows the analysis to focus on industries that better capture **distinct regional economic activity**.

After these exclusions, **12 industries remained and were used in the sector mapping and specialization analysis**:

- Farm compensation  
- Mining  
- Utilities  
- Construction  
- Manufacturing  
- Information  
- Professional, scientific, and technical services  
- Finance and insurance  
- Management of companies and enterprises  
- Arts, entertainment, and recreation  
- Accommodation and food services  

All sector shares and Location Quotients were computed using **only these selected industries**, ensuring that the specialization measure reflects industries that vary meaningfully across states.

### Industry Specialization Using Location Quotients

We classify states by their dominant industry using a **Location Quotient (LQ)**.  
The goal is to identify industries that are **more concentrated in a state relative to the national economy**.

Instead of selecting the largest sector in a state, we compare a sector’s importance in that state to its importance in the U.S. overall.

---

### Step 1 — Compute the sector share within each state

state_share = sector_compensation_in_state / total_compensation_in_state

```python
sector_df = (
    df.groupby(["state","sector"])["compensation"]
    .sum()
    .reset_index()
)

sector_df["state_total"] = sector_df.groupby("state")["compensation"].transform("sum")
sector_df["state_share"] = sector_df["compensation"] / sector_df["state_total"]
```

---

### Step 2 — Compute the national sector share

national_share = total_sector_compensation_US / total_compensation_US

```python
national = sector_df.groupby("sector")["compensation"].sum().reset_index()

national_total = national["compensation"].sum()
national["national_share"] = national["compensation"] / national_total
```

---

### Step 3 — Compute the Location Quotient

LQ = state_share / national_share

```python
sector_df = sector_df.merge(
    national[["sector","national_share"]],
    on="sector"
)

sector_df["LQ"] = sector_df["state_share"] / sector_df["national_share"]
```

Interpretation:

- LQ > 1 → sector is more concentrated than the national average  
- LQ = 1 → sector has average concentration  
- LQ < 1 → sector is less concentrated than average  

---

### Step 4 — Assign the dominant sector

dominant_sector = sector with largest LQ

```python
dominant = sector_df.loc[
    sector_df.groupby("state")["LQ"].idxmax()
]

dominant = dominant[["state","sector","LQ"]]

print(dominant.sort_values("state"))
```

Each state is classified by the sector where it shows the **highest relative specialization** compared to the national economy.

### Data cleaning

In [1]:
pip install pandas requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd

df = pd.read_csv(
    r"C:\Users\nicxn\Documents\Github\Real Estate Data Science Project\Charlotte\SAINC6_2023.csv",
    skiprows=3   # skip BEA header lines
)

print(df.columns)

Index(['GeoFIPS', 'GeoName', 'LineCode', 'Description', '2023'], dtype='object')


In [3]:
df

,GeoFIPS,GeoName,LineCode,Description,2023
0,01000,Alabama,NaN,Compensation of employees by industry (thousan...,NaN
1,01000,Alabama,81.0,Farm compensation,274871
2,01000,Alabama,200.0,"Mining, quarrying, and oil and gas extra...",710120
3,01000,Alabama,300.0,Utilities,2169284
4,01000,Alabama,400.0,Construction,8359728
...,...,...,...,...,...
999,56000,Wyoming,2000.0,Government and government enterprises,6017220
1000,Legend/Footnotes,NaN,NaN,NaN,NaN
1001,Note. All dollar estimates are in thousands of...,NaN,NaN,NaN,NaN
1002,(D) Not shown to avoid disclosure of confident...,NaN,NaN,NaN,NaN


In [4]:
df["2023"] = (
    df["2023"]
    .astype(str)
    .str.replace(",", "")           # remove commas
    .replace(["(D)", "(L)", "(NA)", ""], "0")
)

df["2023"] = pd.to_numeric(df["2023"], errors="coerce").fillna(0)

In [5]:
df = df[df["LineCode"].notna()]

In [6]:
print(df)

    GeoFIPS  GeoName  LineCode  \
1     01000  Alabama      81.0   
2     01000  Alabama     200.0   
3     01000  Alabama     300.0   
4     01000  Alabama     400.0   
5     01000  Alabama     500.0   
..      ...      ...       ...   
995   56000  Wyoming    1500.0   
996   56000  Wyoming    1600.0   
997   56000  Wyoming    1700.0   
998   56000  Wyoming    1800.0   
999   56000  Wyoming    2000.0   

                                           Description        2023  
1                                    Farm compensation    274871.0  
2          Mining, quarrying, and oil and gas extra...    710120.0  
3                                            Utilities   2169284.0  
4                                         Construction   8359728.0  
5                                        Manufacturing  24364815.0  
..                                                 ...         ...  
995                               Educational services    141808.0  
996                  Health care and so

In [7]:
df = df[["GeoName","Description","2023"]]

df.columns = ["state","industry","compensation"]

In [8]:
df

,state,industry,compensation
1,Alabama,Farm compensation,274871.0
2,Alabama,"Mining, quarrying, and oil and gas extra...",710120.0
3,Alabama,Utilities,2169284.0
4,Alabama,Construction,8359728.0
5,Alabama,Manufacturing,24364815.0
...,...,...,...
995,Wyoming,Educational services,141808.0
996,Wyoming,Health care and social assistance,1687118.0
997,Wyoming,"Arts, entertainment, and recreation",170533.0
998,Wyoming,Accommodation and food services,1263837.0


In [9]:
df.tail()

,state,industry,compensation
995,Wyoming,Educational services,141808.0
996,Wyoming,Health care and social assistance,1687118.0
997,Wyoming,"Arts, entertainment, and recreation",170533.0
998,Wyoming,Accommodation and food services,1263837.0
999,Wyoming,Government and government enterprises,6017220.0


In [11]:
import os
os.getcwd()

'c:\\Users\\nicxn\\OneDrive\\Desktop'

In [12]:
df

df.head(20)

,state,industry,compensation
1,Alabama,Farm compensation,274871.0
2,Alabama,"Mining, quarrying, and oil and gas extra...",710120.0
3,Alabama,Utilities,2169284.0
4,Alabama,Construction,8359728.0
5,Alabama,Manufacturing,24364815.0
6,Alabama,Wholesale trade,7838205.0
7,Alabama,Retail trade,10735178.0
8,Alabama,Transportation and warehousing,5666331.0
9,Alabama,Information,2348798.0
10,Alabama,Finance and insurance,8898855.0


In [13]:
df["industry"] = df["industry"].str.strip()

sorted(df["industry"].unique())

df["industry"].str.startswith(" ").sum()

np.int64(0)

In [14]:
drop_industries = [
"Real estate and rental and leasing",
"Government and government enterprises"
]

df = df[~df["industry"].isin(drop_industries)]

df = df.reset_index(drop=True)


In [15]:
print(df.head(18))

      state                                           industry  compensation
0   Alabama                                  Farm compensation      274871.0
1   Alabama      Mining, quarrying, and oil and gas extraction      710120.0
2   Alabama                                          Utilities     2169284.0
3   Alabama                                       Construction     8359728.0
4   Alabama                                      Manufacturing    24364815.0
5   Alabama                                    Wholesale trade     7838205.0
6   Alabama                                       Retail trade    10735178.0
7   Alabama                     Transportation and warehousing     5666331.0
8   Alabama                                        Information     2348798.0
9   Alabama                              Finance and insurance     8898855.0
10  Alabama   Professional, scientific, and technical services    14019325.0
11  Alabama            Management of companies and enterprises     2341461.0

In [ ]:
# industry to sector mapping
industry_map = {

    # Energy
    "Farm compensation": "Energy",
    "Mining": "Energy",
    "Utilities": "Energy",

    # Manufacturing
    "Manufacturing": "Manufacturing",

    # Construction
    "Construction": "Construction",

    # Tech
    "Information": "Tech",
    "Professional, scientific, and technical services": "Tech",

    # Finance
    "Finance and insurance": "Finance",
    "Management of companies and enterprises": "Finance",

    # Tourism
    "Arts, entertainment, and recreation": "Tourism",
    "Accommodation and food services": "Tourism"
}

In [ ]:
# apply the mapping
df["sector"] = df["industry"].map(industry_map)
df = df.dropna(subset=["sector"])

In [ ]:
# aggregate compensation
sector_df = (
    df.groupby(["state","sector"])["compensation"]
    .sum()
    .reset_index()
)

In [ ]:
# compute state sector shares
sector_df["state_total"] = sector_df.groupby("state")["compensation"].transform("sum")

sector_df["state_share"] = sector_df["compensation"] / sector_df["state_total"]

In [ ]:
# compute national sector shares
national = sector_df.groupby("sector")["compensation"].sum().reset_index()

national_total = national["compensation"].sum()

national["national_share"] = national["compensation"] / national_total

In [ ]:
# compute location quotients
sector_df = sector_df.merge(
    national[["sector","national_share"]],
    on="sector"
)

sector_df["LQ"] = sector_df["state_share"] / sector_df["national_share"]

In [ ]:
# classify states
dominant = sector_df.loc[
    sector_df.groupby("state")["LQ"].idxmax()
]

dominant = dominant[["state","sector","LQ"]]

In [ ]:
# print stratified states
print(dominant.sort_values("state"))

              state         sector        LQ
3           Alabama  Manufacturing  1.730617
7            Alaska         Energy  2.308615
12          Arizona   Construction  1.440616
19         Arkansas         Energy  1.801208
28       California           Tech  1.300359
34         Colorado           Tech  1.310699
38      Connecticut        Finance  1.573513
44         Delaware        Finance  1.784886
53          Florida        Tourism  1.560459
58          Georgia           Tech  1.024099
65           Hawaii        Tourism  3.191314
67            Idaho         Energy  2.430912
75         Illinois  Manufacturing  1.141737
81          Indiana  Manufacturing  2.171718
87             Iowa  Manufacturing  1.756506
93           Kansas  Manufacturing  1.589197
99         Kentucky  Manufacturing  1.834900
102       Louisiana   Construction  1.780106
113           Maine        Tourism  1.333419
118        Maryland           Tech  1.322773
124   Massachusetts           Tech  1.364176
129       

In [ ]:
# print distribution of sectors
print(dominant["sector"].value_counts())

sector
Energy           14
Manufacturing    13
Finance           8
Tech              7
Construction      4
Tourism           4
Name: count, dtype: int64


In [99]:
# classify each state by the sector with the highest LQ
dominant = sector_df.loc[
    sector_df.groupby("state")["LQ"].idxmax()
]

dominant = dominant[["state", "sector", "LQ"]].sort_values("state")

# print the table of stratified states
print(dominant)

              state         sector        LQ
3           Alabama  Manufacturing  1.730617
7            Alaska         Energy  2.308615
12          Arizona   Construction  1.440616
19         Arkansas         Energy  1.801208
28       California           Tech  1.300359
34         Colorado           Tech  1.310699
38      Connecticut        Finance  1.573513
44         Delaware        Finance  1.784886
53          Florida        Tourism  1.560459
58          Georgia           Tech  1.024099
65           Hawaii        Tourism  3.191314
67            Idaho         Energy  2.430912
75         Illinois  Manufacturing  1.141737
81          Indiana  Manufacturing  2.171718
87             Iowa  Manufacturing  1.756506
93           Kansas  Manufacturing  1.589197
99         Kentucky  Manufacturing  1.834900
102       Louisiana   Construction  1.780106
113           Maine        Tourism  1.333419
118        Maryland           Tech  1.322773
124   Massachusetts           Tech  1.364176
129       

In [100]:
# sort states by sector first, then alphabetically by state
sorted_states = dominant[["state","sector"]].sort_values(["sector","state"])

print(sorted_states)

              state         sector
12          Arizona   Construction
102       Louisiana   Construction
252           Texas   Construction
258            Utah   Construction
7            Alaska         Energy
19         Arkansas         Energy
67            Idaho         Energy
139     Mississippi         Energy
151         Montana         Energy
157        Nebraska         Energy
181      New Mexico         Energy
199    North Dakota         Energy
211        Oklahoma         Energy
217          Oregon         Energy
241    South Dakota         Energy
265         Vermont         Energy
283   West Virginia         Energy
295         Wyoming         Energy
38      Connecticut        Finance
44         Delaware        Finance
134       Minnesota        Finance
176      New Jersey        Finance
188        New York        Finance
194  North Carolina        Finance
224    Pennsylvania        Finance
230    Rhode Island        Finance
3           Alabama  Manufacturing
75         Illinois 

In [101]:
for sector, group in dominant.groupby("sector"):
    print(f"\n{sector}")
    print(group["state"].sort_values().tolist())


Construction
['Arizona', 'Louisiana', 'Texas', 'Utah']

Energy
['Alaska', 'Arkansas', 'Idaho', 'Mississippi', 'Montana', 'Nebraska', 'New Mexico', 'North Dakota', 'Oklahoma', 'Oregon', 'South Dakota', 'Vermont', 'West Virginia', 'Wyoming']

Finance
['Connecticut', 'Delaware', 'Minnesota', 'New Jersey', 'New York', 'North Carolina', 'Pennsylvania', 'Rhode Island']

Manufacturing
['Alabama', 'Illinois', 'Indiana', 'Iowa', 'Kansas', 'Kentucky', 'Michigan', 'Missouri', 'New Hampshire', 'Ohio', 'South Carolina', 'Tennessee', 'Wisconsin']

Tech
['California', 'Colorado', 'Georgia', 'Maryland', 'Massachusetts', 'Virginia', 'Washington']

Tourism
['Florida', 'Hawaii', 'Maine', 'Nevada']


In [102]:
# group states by sector and create a table
grouped = (
    dominant.groupby("sector")["state"]
    .apply(lambda x: sorted(x.tolist()))
    .reset_index()
)

# count number of states in each sector
grouped["num_states"] = grouped["state"].apply(len)

# format the states as a readable string
grouped["states"] = grouped["state"].apply(lambda x: ", ".join(x))

# reorder columns for display
grouped = grouped[["sector","num_states","states"]]

print(grouped)

          sector  num_states  \
0   Construction           4   
1         Energy          14   
2        Finance           8   
3  Manufacturing          13   
4           Tech           7   
5        Tourism           4   

                                              states  
0                    Arizona, Louisiana, Texas, Utah  
1  Alaska, Arkansas, Idaho, Mississippi, Montana,...  
2  Connecticut, Delaware, Minnesota, New Jersey, ...  
3  Alabama, Illinois, Indiana, Iowa, Kansas, Kent...  
4  California, Colorado, Georgia, Maryland, Massa...  
5                     Florida, Hawaii, Maine, Nevada  


In [103]:
for _, row in grouped.iterrows():
    print(f"{row['sector']} ({row['num_states']} states): {row['states']}")

Construction (4 states): Arizona, Louisiana, Texas, Utah
Energy (14 states): Alaska, Arkansas, Idaho, Mississippi, Montana, Nebraska, New Mexico, North Dakota, Oklahoma, Oregon, South Dakota, Vermont, West Virginia, Wyoming
Finance (8 states): Connecticut, Delaware, Minnesota, New Jersey, New York, North Carolina, Pennsylvania, Rhode Island
Manufacturing (13 states): Alabama, Illinois, Indiana, Iowa, Kansas, Kentucky, Michigan, Missouri, New Hampshire, Ohio, South Carolina, Tennessee, Wisconsin
Tech (7 states): California, Colorado, Georgia, Maryland, Massachusetts, Virginia, Washington
Tourism (4 states): Florida, Hawaii, Maine, Nevada
